In [4]:
import re
import glob
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import xarray as xr


# ============================================================
# Settings
# ============================================================

BASE_DIR = Path("/nird/datalake/NS11071K/users/yongyub/Observation/ORAS5")

SST_DIR = BASE_DIR / "sea_surface_temperature"
WWV_DIR = BASE_DIR / "depth_of_20_c_isotherm"

OUT_DIR = BASE_DIR / "XRO_input"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START = 1998
YEAR_END   = 2025

SST_VAR = "sosstsst"
WWV_VAR = "so20chgt"

# User-requested preprocessing
CLIM_START = f"{YEAR_START}-01-01"
CLIM_END   = f"{YEAR_END}-12-31"

TREND_START = CLIM_START
TREND_END   = CLIM_END

# 1 = linear trend removal
# 2 = quadratic trend removal, closer to the original XRO paper method
TREND_DEGREE = 1

OUT_FILE = OUT_DIR / f"ORAS5_XRO_input_indices_{YEAR_START}_{YEAR_END}_linear_detrended.nc"


# ============================================================
# Index definitions
# longitude convention: 0-360 degE
# ============================================================

SST_BOXES = {
    # ENSO SST index
    "TENSO":  {"lonw": 190, "lone": 240, "lats":  -5, "latn":   5},  # Niño3.4: 170W-120W

    # Pacific modes
    "NPMM":   {"lonw": 200, "lone": 240, "lats":  10, "latn":  25},  # 160W-120W
    "SPMM":   {"lonw": 250, "lone": 270, "lats": -25, "latn": -15},  # 110W-90W

    # Indian Ocean
    "IOB":    {"lonw":  40, "lone": 100, "lats": -20, "latn":  20},
    "IOD1":   {"lonw":  50, "lone":  70, "lats": -10, "latn":  10},
    "IOD2":   {"lonw":  90, "lone": 110, "lats": -10, "latn":   0},
    "SIOD1":  {"lonw":  65, "lone":  85, "lats": -25, "latn": -10},
    "SIOD2":  {"lonw":  90, "lone": 120, "lats": -30, "latn": -10},

    # Atlantic Ocean
    "TNA":    {"lonw": 305, "lone": 345, "lats":   5, "latn":  25},  # 55W-15W
    "ATL3":   {"lonw": 340, "lone": 360, "lats":  -3, "latn":   3},  # 20W-0E
    "SASD1":  {"lonw": 300, "lone": 360, "lats": -45, "latn": -35},  # 60W-0E
    "SASD2":  {"lonw": 320, "lone": 380, "lats": -30, "latn": -20},  # 40W-20E, crosses 0E
}

WWV_BOX = {
    "WWV": {"lonw": 120, "lone": 280, "lats": -5, "latn": 5}  # 120E-80W
}

XRO_MODE_ORDER = [
    "TENSO",
    "WWV",
    "NPMM",
    "SPMM",
    "IOB",
    "IOD",
    "SIOD",
    "TNA",
    "ATL3",
    "SASD",
]


# ============================================================
# Helper functions
# ============================================================

def parse_year_month(path):
    """
    Parse filenames like:
      sea_surface_temperature_1998_01.nc
      depth_of_20_c_isotherm_1998_01.nc
    """
    name = Path(path).name
    m = re.search(r"_(\d{4})_(\d{2})\.nc$", name)
    if m is None:
        return None
    return int(m.group(1)), int(m.group(2))


def find_monthly_files(data_dir, prefix, year_start, year_end):
    files = sorted(glob.glob(str(data_dir / f"{prefix}_*.nc")))

    records = []
    for f in files:
        parsed = parse_year_month(f)
        if parsed is None:
            continue

        year, month = parsed
        if year_start <= year <= year_end:
            records.append(
                {
                    "file": f,
                    "year": year,
                    "month": month,
                    "time": np.datetime64(f"{year:04d}-{month:02d}-15"),
                }
            )

    records = sorted(records, key=lambda r: (r["year"], r["month"]))
    return records


def lon_to_360(lon):
    return (lon + 360.0) % 360.0


def make_box_mask(nav_lon, nav_lat, box):
    """
    Create boolean mask on ORAS5 curvilinear grid.

    box longitude is in 0-360 convention.
    Handles wrap-around if lone > 360 or lone < lonw.
    """
    lon360 = lon_to_360(nav_lon)
    lat = nav_lat

    lonw = box["lonw"]
    lone = box["lone"]
    lats = box["lats"]
    latn = box["latn"]

    lonw_mod = lonw % 360.0
    lone_mod = lone % 360.0

    if (lone <= 360) and (lonw <= lone):
        if lone == 360:
            # Include values near 0E as well
            mask_lon = (lon360 >= lonw_mod) | (lon360 <= 0.0)
        else:
            mask_lon = (lon360 >= lonw_mod) & (lon360 <= lone_mod)
    else:
        # Crosses 0E
        mask_lon = (lon360 >= lonw_mod) | (lon360 <= lone_mod)

    mask_lat = (lat >= lats) & (lat <= latn)

    return mask_lon & mask_lat


def clean_oras5_field(da):
    """
    Convert ORAS5 fill/missing values to NaN.
    """
    arr = da.values.astype(np.float64)

    fill_value = da.attrs.get("_FillValue", None)
    missing_value = da.attrs.get("missing_value", None)

    if fill_value is not None:
        arr = np.where(arr == float(fill_value), np.nan, arr)

    if missing_value is not None:
        arr = np.where(arr == float(missing_value), np.nan, arr)

    # ORAS5 fill values are huge
    arr = np.where(arr > 1e20, np.nan, arr)

    return arr


def weighted_area_mean_2d(arr, mask, weight):
    """
    Area-weighted mean using latitude weight.
    Missing values are excluded from denominator.
    """
    valid = mask & np.isfinite(arr)

    if valid.sum() == 0:
        return np.nan

    num = np.nansum(arr[valid] * weight[valid])
    den = np.nansum(weight[valid])

    if den <= 0:
        return np.nan

    return num / den


def remove_monthly_climatology(x, clim_start, clim_end):
    """
    x: DataArray(time, mode)
    """
    x_ref = x.sel(time=slice(clim_start, clim_end))
    clim = x_ref.groupby("time.month").mean("time", skipna=True)
    anom = x.groupby("time.month") - clim

    return anom, clim


def polynomial_detrend(x, trend_start, trend_end, degree=1):
    """
    Remove polynomial trend independently for each mode.

    x: DataArray(time, mode)
    """
    x_fit = x.sel(time=slice(trend_start, trend_end))

    time_all = pd.to_datetime(x["time"].values)
    time_fit = pd.to_datetime(x_fit["time"].values)

    t0 = time_fit[0]
    t_all = ((time_all - t0) / np.timedelta64(1, "D")) / 365.25
    t_fit = ((time_fit - t0) / np.timedelta64(1, "D")) / 365.25

    detrended = xr.full_like(x, np.nan)
    trend = xr.full_like(x, np.nan)

    coeffs_all = []

    for mode in x["mode"].values:
        y_fit = x_fit.sel(mode=mode).values.astype(float)
        ok = np.isfinite(y_fit)

        if ok.sum() < degree + 2:
            coeffs = np.full(degree + 1, np.nan)
            coeffs_all.append(coeffs)
            continue

        coeffs = np.polyfit(t_fit[ok], y_fit[ok], deg=degree)
        tr_all = np.polyval(coeffs, t_all)

        y_all = x.sel(mode=mode).values.astype(float)

        trend.loc[dict(mode=mode)] = tr_all
        detrended.loc[dict(mode=mode)] = y_all - tr_all

        coeffs_all.append(coeffs)

    coeff_dim = np.arange(degree + 1)

    coeff_da = xr.DataArray(
        np.asarray(coeffs_all),
        dims=("mode", "poly_coeff"),
        coords={
            "mode": x["mode"].values,
            "poly_coeff": coeff_dim,
        },
        name="trend_coefficients",
        attrs={
            "description": (
                "Polynomial coefficients returned by numpy.polyfit. "
                "For degree=1, coeff[0] is slope per year and coeff[1] is intercept."
            ),
            "trend_time_origin": str(t0),
        },
    )

    return detrended, trend, coeff_da


def standardize(x, ref_start, ref_end):
    """
    Standardize each mode using reference period.
    """
    x_ref = x.sel(time=slice(ref_start, ref_end))

    mean = x_ref.mean("time", skipna=True)
    std = x_ref.std("time", skipna=True)

    z = (x - mean) / std

    return z, mean, std


# ============================================================
# Find files
# ============================================================

sst_records = find_monthly_files(
    SST_DIR,
    "sea_surface_temperature",
    YEAR_START,
    YEAR_END,
)

wwv_records = find_monthly_files(
    WWV_DIR,
    "depth_of_20_c_isotherm",
    YEAR_START,
    YEAR_END,
)

expected_n = (YEAR_END - YEAR_START + 1) * 12

print("SST files:", len(sst_records))
print("WWV files:", len(wwv_records))
print("Expected :", expected_n)

if len(sst_records) != expected_n:
    print(f"WARNING: expected {expected_n} SST files, found {len(sst_records)}")

if len(wwv_records) != expected_n:
    print(f"WARNING: expected {expected_n} WWV files, found {len(wwv_records)}")

# Make sure SST and WWV months match
sst_keys = [(r["year"], r["month"]) for r in sst_records]
wwv_keys = [(r["year"], r["month"]) for r in wwv_records]

if sst_keys != wwv_keys:
    raise ValueError("SST and WWV monthly file lists do not match.")


# ============================================================
# Build ORAS5 grid masks from first SST file
# ============================================================

with xr.open_dataset(sst_records[0]["file"], decode_times=False, mask_and_scale=False) as ds0:
    nav_lat = ds0["nav_lat"].values.astype(np.float64)
    nav_lon = ds0["nav_lon"].values.astype(np.float64)

# Latitude weight on curvilinear grid
weight = np.cos(np.deg2rad(nav_lat))
weight = np.where(np.isfinite(nav_lat), weight, np.nan)

sst_masks = {
    name: make_box_mask(nav_lon, nav_lat, box)
    for name, box in SST_BOXES.items()
}

wwv_masks = {
    name: make_box_mask(nav_lon, nav_lat, box)
    for name, box in WWV_BOX.items()
}

print("\nMask grid counts:")
for name, mask in {**sst_masks, **wwv_masks}.items():
    print(f"{name:6s}: {int(np.sum(mask))}")


# ============================================================
# Calculate raw monthly area-mean indices
# ============================================================

raw_component_names = [
    "TENSO",
    "WWV",
    "NPMM",
    "SPMM",
    "IOB",
    "IOD1",
    "IOD2",
    "SIOD1",
    "SIOD2",
    "TNA",
    "ATL3",
    "SASD1",
    "SASD2",
]

raw_components = np.full(
    (len(sst_records), len(raw_component_names)),
    np.nan,
    dtype=np.float64,
)

component_index = {name: i for i, name in enumerate(raw_component_names)}
times = np.array(
    [np.datetime64(r["time"], "ns") for r in sst_records],
    dtype="datetime64[ns]"
)
for it, (sst_rec, wwv_rec) in enumerate(zip(sst_records, wwv_records)):

    year = sst_rec["year"]
    month = sst_rec["month"]

    if (it % 12 == 0) or (it == len(sst_records) - 1):
        print(f"[{it+1:03d}/{len(sst_records):03d}] processing {year}-{month:02d}")

    # SST-based indices
    with xr.open_dataset(sst_rec["file"], decode_times=False, mask_and_scale=False) as ds_sst:
        sst = clean_oras5_field(ds_sst[SST_VAR].isel(time_counter=0))

        for name, mask in sst_masks.items():
            raw_components[it, component_index[name]] = weighted_area_mean_2d(
                sst,
                mask,
                weight,
            )

    # WWV / 20C-isotherm-depth index
    with xr.open_dataset(wwv_rec["file"], decode_times=False, mask_and_scale=False) as ds_wwv:
        wwv = clean_oras5_field(ds_wwv[WWV_VAR].isel(time_counter=0))

        raw_components[it, component_index["WWV"]] = weighted_area_mean_2d(
            wwv,
            wwv_masks["WWV"],
            weight,
        )


raw_comp_da = xr.DataArray(
    raw_components,
    dims=("time", "component"),
    coords={
        "time": times,
        "component": raw_component_names,
    },
    name="raw_component_index",
    attrs={
        "long_name": "Raw area-weighted ORAS5 component indices",
        "description": (
            "Area-weighted monthly mean indices before climatology/trend removal. "
            "SST components are in degC; WWV is depth of 20C isotherm in m."
        ),
    },
)


# ============================================================
# Build final XRO raw indices
# ============================================================

def comp(name):
    return raw_comp_da.sel(component=name).values

raw_final_values = np.column_stack(
    [
        comp("TENSO"),
        comp("WWV"),
        comp("NPMM"),
        comp("SPMM"),
        comp("IOB"),
        comp("IOD1") - comp("IOD2"),
        comp("SIOD1") - comp("SIOD2"),
        comp("TNA"),
        comp("ATL3"),
        comp("SASD1") - comp("SASD2"),
    ]
)

raw_final = xr.DataArray(
    raw_final_values,
    dims=("time", "mode"),
    coords={
        "time": raw_comp_da["time"].values,
        "mode": XRO_MODE_ORDER,
    },
    name="raw_index",
    attrs={
        "long_name": "Raw ORAS5 XRO input indices",
        "description": (
            "Raw area-weighted indices for XRO state variables. "
            "IOD = IOD1 - IOD2, SIOD = SIOD1 - SIOD2, SASD = SASD1 - SASD2."
        ),
        "units_note": "SST-based modes are degC; WWV is m.",
    },
)

raw_final = raw_final.transpose("time", "mode")
raw_final.name = "raw_index"
raw_final.attrs.update(
    {
        "long_name": "Raw ORAS5 XRO input indices",
        "description": (
            "Raw area-weighted indices for XRO state variables. "
            "IOD = IOD1 - IOD2, SIOD = SIOD1 - SIOD2, SASD = SASD1 - SASD2."
        ),
        "units_note": "SST-based modes are degC; WWV is m.",
    }
)


# ============================================================
# XRO-style preprocessing
# ============================================================

monthly_anom, monthly_climatology = remove_monthly_climatology(
    raw_final,
    CLIM_START,
    CLIM_END,
)

xro_input, removed_trend, trend_coefficients = polynomial_detrend(
    monthly_anom,
    TREND_START,
    TREND_END,
    degree=TREND_DEGREE,
)

xro_input.name = "xro_input"
xro_input.attrs.update(
    {
        "long_name": "XRO input indices",
        "description": (
            "Raw indices after removing monthly climatology and polynomial trend. "
            f"Trend degree = {TREND_DEGREE}."
        ),
        "units_note": "SST-based modes are degC anomalies; WWV is m anomaly.",
    }
)

xro_input_standardized, std_mean, std_dev = standardize(
    xro_input,
    CLIM_START,
    CLIM_END,
)

xro_input_standardized.name = "xro_input_standardized"
xro_input_standardized.attrs.update(
    {
        "long_name": "Standardized XRO input indices",
        "description": (
            "XRO input indices standardized independently for each mode "
            "using the reference period."
        ),
        "units": "standard deviation",
    }
)

monthly_anom.name = "monthly_anomaly"
monthly_anom.attrs.update(
    {
        "long_name": "Monthly climatology removed ORAS5 XRO indices",
    }
)

monthly_climatology.name = "monthly_climatology"
removed_trend.name = "removed_trend"
std_mean.name = "standardization_mean"
std_dev.name = "standardization_std"


# ============================================================
# Output Dataset
# ============================================================

out_ds = xr.Dataset(
    data_vars={
        "raw_index": raw_final,
        "monthly_anomaly": monthly_anom,
        "removed_trend": removed_trend,
        "xro_input": xro_input,
        "xro_input_standardized": xro_input_standardized,
        "monthly_climatology": monthly_climatology,
        "trend_coefficients": trend_coefficients,
        "standardization_mean": std_mean,
        "standardization_std": std_dev,
        "raw_component_index": raw_comp_da,
    },
    attrs={
        "title": "ORAS5 XRO input indices",
        "source": "ORAS5 sea surface temperature and depth of 20C isotherm",
        "sst_source_directory": str(SST_DIR),
        "wwv_source_directory": str(WWV_DIR),
        "output_file": str(OUT_FILE),
        "period": f"{YEAR_START}-{YEAR_END}",
        "method": (
            "Area-weighted means were computed using cos(nav_lat) weights on the ORAS5 grid. "
            "For XRO input, monthly climatology was removed first, then a polynomial trend "
            f"of degree {TREND_DEGREE} was removed independently for each mode."
        ),
        "climatology_period": f"{CLIM_START} to {CLIM_END}",
        "trend_fit_period": f"{TREND_START} to {TREND_END}",
        "mode_order": ", ".join(XRO_MODE_ORDER),
        "component_note": (
            "IOD = IOD1 - IOD2; SIOD = SIOD1 - SIOD2; SASD = SASD1 - SASD2. "
            "raw_component_index stores the underlying component boxes."
        ),
        "time_coordinate_note": (
            "Time coordinate is based on filename year/month and assigned to the 15th day of each month."
        ),
        "created_by": "custom Python/xarray script",
        "date_created": date.today().isoformat(),
    },
)

out_ds["time"].attrs.update(
    {
        "long_name": "time",
        "description": "Monthly representative time based on filename, set to day 15.",
    }
)

out_ds["mode"].attrs.update(
    {
        "long_name": "XRO state variable name",
    }
)

out_ds["component"].attrs.update(
    {
        "long_name": "Raw regional component index name",
    }
)

out_ds["month"].attrs.update(
    {
        "long_name": "calendar month",
    }
)


# ============================================================
# Save
# ============================================================

encoding = {
    "raw_index": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "monthly_anomaly": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "removed_trend": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "xro_input": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "xro_input_standardized": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "monthly_climatology": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "removed_trend": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "trend_coefficients": {
        "dtype": "float64",
        "_FillValue": np.float64(-999.0),
    },
    "standardization_mean": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
    },
    "standardization_std": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
    },
    "raw_component_index": {
        "dtype": "float32",
        "_FillValue": np.float32(-999.0),
        "zlib": True,
        "complevel": 4,
    },
    "time": {
        "dtype": "float64",
        "units": "days since 1900-01-01",
        "calendar": "gregorian",
    },
}

if OUT_FILE.exists():
    OUT_FILE.unlink()

out_ds.to_netcdf(OUT_FILE, encoding=encoding, engine="netcdf4")

print(f"\nSaved: {OUT_FILE}")
print(out_ds)

SST files: 336
WWV files: 336
Expected : 336

Mask grid counts:
TENSO : 8241
NPMM  : 10130
SPMM  : 3483
IOB   : 39609
IOD1  : 6561
IOD2  : 3321
SIOD1 : 5229
SIOD2 : 10285
TNA   : 13332
ATL3  : 2025
SASD1 : 12532
SASD2 : 10604
WWV   : 26281
[001/336] processing 1998-01
[013/336] processing 1999-01
[025/336] processing 2000-01
[037/336] processing 2001-01
[049/336] processing 2002-01
[061/336] processing 2003-01
[073/336] processing 2004-01
[085/336] processing 2005-01
[097/336] processing 2006-01
[109/336] processing 2007-01
[121/336] processing 2008-01
[133/336] processing 2009-01
[145/336] processing 2010-01
[157/336] processing 2011-01
[169/336] processing 2012-01
[181/336] processing 2013-01
[193/336] processing 2014-01
[205/336] processing 2015-01
[217/336] processing 2016-01
[229/336] processing 2017-01
[241/336] processing 2018-01
[253/336] processing 2019-01
[265/336] processing 2020-01
[277/336] processing 2021-01
[289/336] processing 2022-01
[301/336] processing 2023-01
[313/3